# 40.20 Оператор переноса известных тканевых вкладов в ТТРКГ

Временные функции мягких тканей и лёгкого из условной двуслойной задачи серии
33 переносятся на точную временную сетку каждого ансамбля ТТРКГ через
конфигурационно-специфичный FEM-оператор. Это предварительная модельная оценка
известных вкладов, а не универсальные свойства тканей и не сердечный сигнал.
Все остальные анатомические регионы явно сохраняются как немоделированные
динамические вклады.


In [ ]:
import hashlib
import json
import os
from pathlib import Path

import numpy as np

from ttrkg_analysis import apply_fractional_operator

test = apply_fractional_operator([0.6, -0.2], np.array([[0.01, -0.01], [-0.02, 0.01]]))
np.testing.assert_allclose(test, [0.010, -0.008])
REAL_MODE = os.environ.get("KALMYKOV_RUN_REAL", "0") == "1"
print("40.20 synthetic_self_test: passed")


In [ ]:
KEY_FIELDS = ['experiment_id', 'subject_id', 'record_id', 'configuration_id', 'montage_id', 'side_montage_id', 'side_size_mm', 'channel_state', 'mode']


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def key(item):
    return tuple(item.get(field) for field in KEY_FIELDS)


def finite_numbers(value):
    if isinstance(value, dict):
        return [number for child in value.values() for number in finite_numbers(child)]
    if isinstance(value, list):
        return [number for child in value for number in finite_numbers(child)]
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return [float(value)]
    return []


if not REAL_MODE:
    print("40.20 real_data_status: blocked_until_series33_measured_fem_and_transfer_contracts")
else:
    target = os.environ.get("KALMYKOV_TARGET_EXPERIMENT")
    if target not in {"exp02", "exp03"}:
        raise RuntimeError("KALMYKOV_TARGET_EXPERIMENT должен быть exp02 или exp03")
    target_config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG" if target == "exp02" else "KALMYKOV_EXP03_CONFIG"]).expanduser().resolve()
    target_config = json.loads(target_config_path.read_text(encoding="utf-8"))
    target_root = Path(target_config["derived_root"]).expanduser().resolve()
    target_dir = target_root / target / "analysis"
    measured_path = target_dir / ("40.01_ttrkg_ensembles.json" if target == "exp02" else "40.11_ttrkg_ensembles.json")
    fem_path = target_dir / ("40.04_fem_operator_checked.json" if target == "exp02" else "40.14_fem_operator_checked.json")
    measured = json.loads(measured_path.read_text(encoding="utf-8"))
    fem = json.loads(fem_path.read_text(encoding="utf-8"))
    if measured.get("status") != "accepted_input_conditional_ttrkg_ensembles" or not measured.get("ensembles"):
        raise RuntimeError("Не принят измеренный артефакт ТТРКГ")
    if fem.get("status") != "accepted_complete_fem_operator_contract" or not fem.get("operators"):
        raise RuntimeError("Не принят полный FEM-оператор")
    measured_by_key = {key(item): item for item in measured["ensembles"]}
    fem_by_key = {key(item): item for item in fem["operators"]}
    if len(measured_by_key) != len(measured["ensembles"]) or len(fem_by_key) != len(fem["operators"]) or set(measured_by_key) != set(fem_by_key):
        raise RuntimeError("Ключи измерений и FEM должны точно совпадать")

    source_config_path = Path(os.environ["KALMYKOV_EXP02_CONFIG"]).expanduser().resolve()
    source_config = json.loads(source_config_path.read_text(encoding="utf-8"))
    source_dir = Path(source_config["derived_root"]).expanduser().resolve() / "exp02" / "analysis"
    static_path = source_dir / "33.01_static.json"
    source_ensemble_path = source_dir / "33.03_ensembles.json"
    pulse_path = source_dir / "33.04_delta_rho.json"
    static = json.loads(static_path.read_text(encoding="utf-8"))
    source_ensemble = json.loads(source_ensemble_path.read_text(encoding="utf-8"))
    pulse = json.loads(pulse_path.read_text(encoding="utf-8"))
    if static.get("status") != "conditional_two_layer_estimate" or source_ensemble.get("status") != "accepted_input_conditional_ensembles" or pulse.get("status") != "conditional_linearized_two_layer_estimate":
        raise RuntimeError("Неподходящий статус артефактов серии 33")
    timing = source_ensemble.get("signal_operator", {})
    if timing.get("timing_calibration_status") != "accepted" or timing.get("group_delay_s") is None:
        raise RuntimeError("Для бокового сигнала не принята групповая задержка")
    side_delay_s = float(timing["group_delay_s"])
    pulse_by_subject_mode = {(item["subject_id"], item["mode"]): item for item in pulse["results"]}
    if len(pulse_by_subject_mode) != len(pulse["results"]):
        raise RuntimeError("Повторный ключ в 33.04")

    transfer_hash = None
    source_for_target = {}
    if target == "exp02":
        source_for_target = {item["subject_id"]: item["subject_id"] for item in measured["ensembles"]}
        status = "same_experiment_conditional_known_tissue_prediction"
    else:
        transfer_spec = target_config.get("transfer_analysis", {})
        if transfer_spec.get("cross_session_status") != "accepted_preliminary" or transfer_spec.get("uncertainty_status") != "accepted" or not transfer_spec.get("uncertainty_manifest"):
            raise RuntimeError("Межсессионный перенос не принят и не снабжён численной моделью неопределённости")
        transfer_path = Path(transfer_spec["uncertainty_manifest"]).expanduser().resolve()
        transfer = json.loads(transfer_path.read_text(encoding="utf-8"))
        if transfer.get("status") != "accepted" or transfer.get("source_experiment") != "exp02" or transfer.get("target_experiment") != "exp03":
            raise RuntimeError("Неподходящий манифест межсессионной неопределённости")
        numbers = finite_numbers(transfer.get("numeric_parameters", {}))
        if not numbers or not np.isfinite(numbers).all() or not transfer.get("provenance"):
            raise RuntimeError("Манифест переноса не содержит численных параметров и происхождения")
        source_for_target = {transfer["target_subject_id"]: transfer["source_subject_id"]}
        transfer_hash = sha256_file(transfer_path)
        status = "preliminary_cross_session_known_tissue_prediction"

    outputs = []
    for item_key, measured_item in measured_by_key.items():
        target_subject = measured_item["subject_id"]
        if target_subject not in source_for_target:
            raise RuntimeError(f"Нет явного соответствия исходного добровольца для {target_subject}")
        source_subject = source_for_target[target_subject]
        source_item = pulse_by_subject_mode.get((source_subject, measured_item["mode"]))
        if source_item is None:
            raise RuntimeError("Нет временных функций 33.04 для требуемого добровольца и режима")
        source_grid = np.asarray(source_item["time_from_r_s"], float) - side_delay_s
        target_grid = np.asarray(measured_item["time_from_r_corrected_s"], float)
        if target_grid[0] < source_grid[0] or target_grid[-1] > source_grid[-1]:
            raise RuntimeError("Целевая временная сетка выходит за поддержку сигнала 33.04")
        estimate = static["subjects"][source_subject]["estimate"]
        rho1 = float(estimate["rho1_ohm_m"])
        rho2 = float(estimate["rho2_inhale_ohm_m"] if measured_item["mode"] == "задержка_вдох" else estimate["rho2_exhale_ohm_m"])
        soft = np.interp(target_grid, source_grid, np.asarray(source_item["delta_rho1_ohm_m"], float) / rho1)
        lung = np.interp(target_grid, source_grid, np.asarray(source_item["delta_rho2_ohm_m"], float) / rho2)
        operator = fem_by_key[item_key]
        names = operator["region_names"]
        if "soft_tissue_wall" not in names or "lung" not in names:
            raise RuntimeError("В полном FEM-операторе нет мягких тканей или лёгкого")
        sensitivity = np.asarray([operator["fractional_sensitivities"][names.index("soft_tissue_wall")], operator["fractional_sensitivities"][names.index("lung")]], float)
        predicted = apply_fractional_operator(sensitivity, np.vstack([soft, lung]))
        outputs.append({
            **{field: measured_item.get(field) for field in KEY_FIELDS},
            "source_subject_id": source_subject, "source_experiment": "exp02", "target_experiment": target,
            "time_from_r_corrected_s": target_grid.tolist(),
            "predicted_known_tissue_fractional_delta_z": predicted.tolist(),
            "modeled_dynamic_regions": ["soft_tissue_wall", "lung"],
            "unmodeled_dynamic_regions": [name for name in names if name not in {"soft_tissue_wall", "lung"}],
            "fem_operator_manifest_sha256": operator["operator_manifest_sha256"],
            "transfer_uncertainty_manifest_sha256": transfer_hash,
            "total_variance_fractional": None,
        })
    if not outputs or {key(item) for item in outputs} != set(measured_by_key):
        raise RuntimeError("Предсказания должны непусто и один-к-одному покрывать измерения")
    artifact = {
        "schema_version": 2, "analysis": "40.20_known_tissue_transfer", "status": status,
        "measured_artifact_sha256": sha256_file(measured_path), "fem_artifact_sha256": sha256_file(fem_path),
        "source_static_sha256": sha256_file(static_path), "source_ensemble_sha256": sha256_file(source_ensemble_path),
        "source_delta_rho_sha256": sha256_file(pulse_path),
        "uncertainty_status": "total_budget_not_propagated_here", "predictions": outputs,
    }
    out_path = target_dir / "40.20_tissue_prediction.json"
    out_path.write_text(json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("40.20 real_data_status: conditional_artifact_written", out_path)


## Граница результата

Даже полный FEM-раздел не означает, что временные изменения всех регионов
известны. Здесь заданы только мягкие ткани и лёгкое. Межсессионный перенос в
эксперимент 3 остаётся предварительным и не заменяет независимую проверку.
